# Аналіз даних ковідного відділення
### Звіт за результатами первинного опрацювання

**Виконавець:** аналітик відділу статистики
**Набір даних:** `Ковід_дані_навчальні.xlsx`, 121 пацієнт, 77 ознак

---

Цей звіт готував колега. Він виконується від початку до кінця без жодної помилки,
усі числа в ньому обчислено правильно з арифметичного погляду.

**У звіті є рівно десять місць, де статистику застосовано до шкали, яка її не
допускає, або де результат отримано з даних, які до цього не придатні.**

Ваше завдання — знайти всі десять, пояснити кожну і показати правильний варіант.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('Ковід_дані_навчальні.xlsx', sheet_name='Лист1', engine='openpyxl')
print('Завантажено:', df.shape[0], 'пацієнтів,', df.shape[1], 'ознак')

Завантажено: 121 пацієнтів, 77 ознак


## 1. Загальна характеристика результату лікування

In [2]:
середній_результат = df['помер/жив'].mean()
print('Середній результат лікування:', round(середній_результат, 2))
print()
print('Оскільки 1 = помер, а 2 = вижив, значення 1.50 означає,')
print('що в середньому пацієнти відділення перебували рівно посередині')
print('між летальним і сприятливим результатом.')

Середній результат лікування: 1.5

Оскільки 1 = помер, а 2 = вижив, значення 1.50 означає,
що в середньому пацієнти відділення перебували рівно посередині
між летальним і сприятливим результатом.


## 2. Демографія: розподіл за статтю

In [3]:
стать = df['Стать'].value_counts()
print('У вибірці представлено', df['Стать'].nunique(), 'статі:')
print(стать.to_string())
print()
print('Найбільша група — «ж» (56 осіб, 46%).')
print('Групи «жін» та «чол» помітно менші, що потребує окремого пояснення.')

У вибірці представлено 4 статі:
Стать
ж      56
ч      34
жін    19
чол    12

Найбільша група — «ж» (56 осіб, 46%).
Групи «жін» та «чол» помітно менші, що потребує окремого пояснення.


## 3. Тяжкість ураження легень за даними КТ

In [4]:
ступінь = df['ступінь КТ'].value_counts().sort_index()
print('Розподіл за ступенем ураження (за зростанням):')
print(ступінь.to_string())
print()
print('Найлегший ступінь у вибірці —', ступінь.index[0] + ',')
print('найважчий —', ступінь.index[-1] + '.')
print()
коди = {назва: i for i, назва in enumerate(ступінь.index)}
середній_ступінь = df['ступінь КТ'].map(коди).mean()
print('Середній ступінь ураження:', round(середній_ступінь, 2),
      '->', ступінь.index[int(round(середній_ступінь))])

Розподіл за ступенем ураження (за зростанням):
ступінь КТ
важкий            9
важкий            1
вкрай важкий      1
вкрай важкий      3
критичний         3
легка             1
легкий            5
легкий            1
сердній           1
серед ступ        6
серед ступеня     1
середній         10

Найлегший ступінь у вибірці — важкий,
найважчий — середній.

Середній ступінь ураження: 5.9 -> легкий


## 4. Обсяг ураження легень у відсотках

In [5]:
кт = pd.to_numeric(df['КТ'], errors='coerce')
print('Середній обсяг ураження легень:', round(100 * кт.mean(), 1), '%')
print('Медіана:', round(100 * кт.median(), 1), '%')
print('Стандартне відхилення:', round(100 * кт.std(), 1), '%')

Середній обсяг ураження легень: 39.6 %
Медіана: 30.0 %
Стандартне відхилення: 22.9 %


## 5. Сатурація при поступленні

In [6]:
s = df['spO2 без дот']
print('Сатурація при поступленні без дотації кисню')
print('  середнє      :', round(s.mean(), 2))
print('  медіана      :', round(s.median(), 2))
print('  мінімум      :', round(s.min(), 2))
print('  максимум     :', round(s.max(), 2))
print()
print('Середня сатурація 82.05% свідчить про виражену дихальну недостатність')
print('у більшості пацієнтів на момент госпіталізації.')

Сатурація при поступленні без дотації кисню
  середнє      : 82.05
  медіана      : 84.0
  мінімум      : 0.9
  максимум     : 98.0

Середня сатурація 82.05% свідчить про виражену дихальну недостатність
у більшості пацієнтів на момент госпіталізації.


## 6. Стабільність дат госпіталізації

In [7]:
дати = pd.to_datetime(df['Поступлення'], errors='coerce').dropna()
як_числа = дати.astype('int64') / 10**9 / 86400          # дні від початку епохи

cv = як_числа.std() / як_числа.mean()
print('Середня дата поступлення:', дати.mean().date())
print('Коефіцієнт варіації дат  :', round(100 * cv, 3), '%')
print()
print('Коефіцієнт варіації менший за 1% означає надзвичайно високу')
print('однорідність потоку госпіталізацій у часі.')

Середня дата поступлення: 2022-01-12
Коефіцієнт варіації дат  : 0.595 %

Коефіцієнт варіації менший за 1% означає надзвичайно високу
однорідність потоку госпіталізацій у часі.


## 7. Зв'язок статі з результатом лікування

In [8]:
стать_код = df['Стать'].astype(str).str.strip().str[0].map({'ч': 0, 'ж': 1})
помер = (df['помер/жив'] == 1).astype(int)

r = стать_код.corr(помер)
print('Коефіцієнт кореляції Пірсона між статтю та летальністю:', round(r, 3))
print()
print('Значення близьке до нуля свідчить про відсутність лінійного зв’язку')
print('між статтю пацієнта та результатом лікування.')

Коефіцієнт кореляції Пірсона між статтю та летальністю: 0.177

Значення близьке до нуля свідчить про відсутність лінійного зв’язку
між статтю пацієнта та результатом лікування.


## 8. Підготовка ознак: нормалізація

In [9]:
числові = df.select_dtypes('number')
нормалізовані = (числові - числові.min()) / (числові.max() - числові.min())

print('Нормалізовано стовпців:', нормалізовані.shape[1])
print('Усі значення приведено до діапазону [0, 1].')
print()
print(нормалізовані[['Вік', 'Л1', 'ФЕРИТ2']].describe().round(3).to_string())

Нормалізовано стовпців: 40
Усі значення приведено до діапазону [0, 1].

           Вік       Л1  ФЕРИТ2
count  121.000  117.000     0.0
mean     0.495    0.275     NaN
std      0.190    0.175     NaN
min      0.000    0.000     NaN
25%      0.375    0.154     NaN
50%      0.516    0.221     NaN
75%      0.609    0.375     NaN
max      1.000    1.000     NaN


## 9. Очищення набору від пропусків

In [10]:
чистий = df.dropna()
print('Було рядків :', len(df))
print('Стало рядків:', len(чистий))
print()
print('Після видалення рядків із пропусками отримано повністю заповнений набір,')
print('придатний для побудови моделі.')

Було рядків : 121
Стало рядків: 0

Після видалення рядків із пропусками отримано повністю заповнений набір,
придатний для побудови моделі.


## 10. Рейтинг важливості ознак

In [11]:
дані = df.select_dtypes('number').copy()
дані['Дата розтину'] = pd.to_datetime(df['Дата розтину'], errors='coerce').astype('int64')
ціль = (df['помер/жив'] == 1).astype(int)

рейтинг = dict()
for c in дані.columns:
    if c == 'помер/жив':
        continue
    r = дані[c].corr(ціль)
    if pd.notna(r):
        рейтинг[c] = abs(r)

топ = pd.Series(рейтинг).sort_values(ascending=False).head(8)
print('Найважливіші ознаки для прогнозування летальності:')
print()
print(топ.round(3).to_string())
print()
print('Найсильніший предиктор — дата розтину (r = 1.00).')
print('Далі йдуть показники лімфоцитів, інтерлейкіну-6 та лейкоцитів.')
print('Рекомендуємо будувати модель на перших п’яти ознаках.')

Найважливіші ознаки для прогнозування летальності:

Дата розтину    1.000
Лім2            0.645
ІЛ6-2           0.616
Л2              0.480
Вік             0.449
П/Я2            0.395
АЛАТ1           0.392
АСАТ1           0.388

Найсильніший предиктор — дата розтину (r = 1.00).
Далі йдуть показники лімфоцитів, інтерлейкіну-6 та лейкоцитів.
Рекомендуємо будувати модель на перших п’яти ознаках.


---
## Висновки звіту

1. Середній результат лікування — 1.50, тобто рівно посередині між летальним і сприятливим.
2. У вибірці представлено чотири статі.
3. Середній ступінь ураження легень — «легкий».
4. Середній обсяг ураження — 39.6%.
5. Середня сатурація при поступленні — 82.05%.
6. Потік госпіталізацій украй однорідний: коефіцієнт варіації дат менший за 1%.
7. Зв’язку між статтю та летальністю не виявлено.
8. Усі числові ознаки успішно нормалізовано.
9. Після очищення отримано повністю заповнений набір даних.
10. Найсильніший предиктор летальності — дата розтину.

---

### Ваше завдання

Знайдіть **десять** місць, де цей звіт помиляється. Для кожного:

* назвіть номер розділу;
* вкажіть, яка саме шкала вимірювання порушена або яке припущення хибне;
* поясніть, чому отримане число не означає того, що написано у висновку;
* наведіть правильний варіант обчислення — або поясніть, чому правильного варіанта
  не існує і від показника треба відмовитися.

Помилки бувають двох типів: **статистика, недопустима для цієї шкали**, і
**число, обчислене на даних, які до цього не придатні**. У звіті є обидва типи.